# Phase 2 — Retrieval

Hybrid (dense + FTS + RRF) vs. dense-only, then reranking, on a handful of hand-written queries scoped to the ingested corpus. Run `01_ingestion.ipynb` first.

In [ ]:
import subprocess, sys, os
REPO_URL = "https://github.com/<your-username>/stance.git"  # TODO: set this
if not os.path.isdir("/kaggle/working/stance"):
    subprocess.run(["git", "clone", REPO_URL, "/kaggle/working/stance"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/kaggle/working/stance[retrieval]"], check=True)
sys.path.insert(0, "/kaggle/working/stance")

In [ ]:
from stance.db import get_engine
from stance.ingest.embed import embed_query
from stance.retrieval.hybrid import hybrid_search
from stance.retrieval.rerank import rerank
from sqlalchemy import text

engine = get_engine()
with engine.connect() as conn:
    aapl_cik = conn.execute(text("SELECT cik FROM companies WHERE ticker='AAPL'")).scalar()

## T5.1 — Section-scoped narrative: item-code prefilter must hold

"What are Apple's supply chain risks?" scoped to Item 1A must return zero MD&A (Item 7) chunks.

In [ ]:
query = "What are Apple's supply chain risks?"
results = hybrid_search(engine, query, embed_query(query), cik=aapl_cik, item_code="1A", top_k=8)
item_codes_returned = {r["item_code"] for r in results}
print(f"Item codes in results: {item_codes_returned}")
assert item_codes_returned <= {"1A"}, "Section prefilter leaked non-Item-1A content!"
for r in results[:3]:
    print(f"  [{r['score']:.4f}] {r['text'][:150]}...")

## Ablation — dense-only vs. RRF vs. RRF+rerank

Hand-label the relevant chunk_ids for each query below (fill in `relevant_ids` after eyeballing your corpus once), then hit@k is computed automatically. Until labelled, this cell just prints candidates for you to inspect.

In [ ]:
ABLATION_QUERIES = [
    "What are Apple's main competitive risks?",
    "remaining performance obligations",
    "Microsoft cloud revenue growth",
]

for q in ABLATION_QUERIES:
    qvec = embed_query(q)
    dense_only = hybrid_search(engine, "", qvec, top_k=8)  # empty query_text neutralises the FTS leg
    rrf = hybrid_search(engine, q, qvec, top_k=8)
    reranked = rerank(q, rrf, top_k=8)

    print(f"\n=== {q!r} ===")
    print(f"dense-only top-1: {dense_only[0]['text'][:100]}..." if dense_only else "dense-only: no results")
    print(f"RRF top-1:        {rrf[0]['text'][:100]}..." if rrf else "RRF: no results")
    print(f"reranked top-1:   {reranked[0]['text'][:100]}..." if reranked else "reranked: no results")